In [ ]:
import torch
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
from mhn import wfp_gram_uniform_init, dynamics_gaussian_init, wfp_gram_dirichlet_init, get_critical_beta_gram_uniform_fixed_point, annealing_uniform_init
from math import log, sqrt, fabs, ceil, floor, atan
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from tqdm import tqdm
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, to_tree
from scipy.spatial.distance import pdist
from itertools import product
plt.style.use('seaborn-custom-whitegrid.mplstyle')
import matplotlib.patheffects as patheffects

In [ ]:
def patterns_shift_and_rms(x, eps=1e-12):
    shift = x.mean(dim=0, keepdim=True)
    x_shifted = x - shift

    rms = x_shifted.norm(dim=1).square().mean().sqrt()
    rms = rms.clamp_min(eps)

    x_norm = x_shifted / rms

    return shift, rms, x_norm

def patterns_sphere(x):
    radii = x.norm(dim=1)
    return radii, x / radii.expand_as(x)

In [ ]:
def get_cs_metric(betas, cosines, N):
    
    weights = torch.softmax(betas[:, None] * cosines.view(-1), dim=-1)
    mean = 1 -  weights @ cosines.view(-1)
    var = torch.sum(((1-cosines.view(-1)[None,:])**2 - mean[:, None]**2) * weights, dim=-1)
    gaussian_part = N/(8*betas**2)
    mean_part = (-1/(4*betas)) * mean
    full_metric = gaussian_part + mean_part + (1/16)*var
    scaled_metric = 1.0 - 2*(betas/N)*mean + ((betas/sqrt(N))**2)*var/2
    return full_metric, mean_part + (1/16)*var, scaled_metric

In [ ]:
def get_entropies(weights):
    entropies = - weights * weights.log()
    entropies[torch.isnan(entropies)] = 0
    return entropies.sum(dim=-1)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    device = 'mps' if torch.backends.mps.is_available() else 'cpu'

## IID patterns

### (1) Many Gram matrices

In [ ]:
torch.manual_seed(11)
N = 32
betas_unif_init = torch.linspace(0, 100, 201)[1:]
num_runs_unif_init = 100

Ks = [16,32,64,128, 256,512, 1024]
all_results_unif_init = {}
all_beta_c_unif_init = {}
for K in Ks:
    beta_cs = []
    all_weights_unif_init = []
    for run in tqdm(range(num_runs_unif_init)):
        raw_patterns = torch.randn(K,N)
        shift, rms, patterns = patterns_shift_and_rms(raw_patterns)
        gram_matrix = patterns @ patterns.t()
        beta_c = get_critical_beta_gram_uniform_fixed_point(gram_matrix)
        beta_cs.append(beta_c)
        weights_unif_init = wfp_gram_uniform_init(betas_unif_init.to(device), gram_matrix.to(device), 3000,verbose=False)
        weights_unif_init = weights_unif_init.cpu()
        all_weights_unif_init.append(weights_unif_init)
    all_weights_unif_init = torch.stack(all_weights_unif_init, dim=0)
    all_results_unif_init[K] = all_weights_unif_init
    all_beta_c_unif_init[K] = torch.tensor(beta_cs)

In [ ]:
np.savez("results/wfp_iid.npz", N=N, Ks=Ks, betas=betas_unif_init, results = all_results_unif_init, beta_cs = all_beta_c_unif_init)

In [ ]:
saved_results = np.load("results/wfp_iid.npz", allow_pickle=True)
N = saved_results['N'].item()
Ks = saved_results['Ks']
betas_unif_init = saved_results['betas']
all_results_unif_init = saved_results['results'].item()
all_beta_c_unif_init = saved_results['beta_cs'].item()

In [ ]:
palette = plt.get_cmap('Set1')
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
for K_idx, K in enumerate(Ks):
    all_weights_unif_init = all_results_unif_init[K]
    mean_weights_unif_init = all_weights_unif_init.mean(dim=0)
    all_entropies_unif_init = get_entropies(all_weights_unif_init)
    mean_entropies_unif_init = all_entropies_unif_init.mean(dim=0)
    quantiles_entropies_unif_init = torch.quantile(all_entropies_unif_init, torch.tensor([0.1, 0.5, 0.9]), dim=0)
    beta_crit_unif_init = K/(1+sqrt(K/N))**2
    ax.plot(betas_unif_init, mean_entropies_unif_init/log(K), color=palette(K_idx))
    ax.axvline(beta_crit_unif_init, linestyle='--',alpha=0.5, label='$\\beta_c$, $K={}$'.format(K), color=palette(K_idx))
ax.set_xscale('log')
ax.set_xlabel('$\\beta$')
ax.set_ylabel('$H/\\log(K)$')
ax.legend(fontsize=10)
fig.savefig('plots/wfp_iid_entropies.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
all_beta_c_ratios = []
for K in Ks:
    beta_crit_unif_init = K/(1+sqrt(K/N))**2
    beta_cs = all_beta_c_unif_init[K]
    all_beta_c_ratios.append((np.log(K)/N, beta_crit_unif_init/beta_cs.mean().item()))
all_beta_c_ratios = np.array(all_beta_c_ratios)
ax.plot(all_beta_c_ratios[:,0], all_beta_c_ratios[:,1], marker='o')
ax.set_xlabel('$\\log(K)/N)$')
ax.set_ylabel('$\\beta_{c, \\rm{th}}$ / $\\beta_{c, \\rm{emp}}$')
fig.savefig('plots/wfp_iid_beta_crit.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 1-step Hierarchical vary $K$ and $B$

### Run

In [ ]:
torch.manual_seed(11)
N = 32
rho0 = 0.1
rho1 = 0.9
betas_unif_init = torch.logspace(0, 3, 201)[1:]
num_runs_unif_init = 100

Ks = [16,32,64,128, 256,512, 1024]
block_frac_ptwos = range(0, int(log(Ks[-1], 2))+1)
all_results_unif_init = {}
all_beta_c_unif_init = {}
for (K, frac_ptwo) in product(Ks, block_frac_ptwos):
    B = K/(2 ** frac_ptwo)
    if B <= 1:
        continue
    B = int(B)
    M = K//B
    if M*B != K:
        continue

    print('K={}, B={}, M={}'.format(K, B, M))
    beta_cs = []
    all_weights_unif_init = []
    for run in tqdm(range(num_runs_unif_init)):
        z_0 = torch.randn(N,)
        z_block = torch.randn(M, N)
        eps = torch.randn(K, N)

        raw_patterns = sqrt(rho0)*z_0 + sqrt(rho1-rho0)* torch.repeat_interleave(z_block, repeats=B, dim=0) + sqrt(1-rho1)*eps
        shift, rms, patterns = patterns_shift_and_rms(raw_patterns)
        gram_matrix = patterns @ patterns.t()
        beta_c = get_critical_beta_gram_uniform_fixed_point(gram_matrix)
        weights_unif_init = wfp_gram_uniform_init(betas_unif_init.to(device), gram_matrix.to(device), 3000,verbose=False)
        weights_unif_init = weights_unif_init.cpu()
        all_weights_unif_init.append(weights_unif_init)
        beta_cs.append(beta_c)
    all_weights_unif_init = torch.stack(all_weights_unif_init, dim=0)
    all_results_unif_init[(K, B)] = all_weights_unif_init
    all_beta_c_unif_init[(K, B)] = torch.tensor(beta_cs)


In [ ]:
np.savez("results/wfp_one_block_rho0={}_rho1={}.npz".format(rho0, rho1), N=N, Ks=Ks, rho0=rho0, rho1=rho1, betas=betas_unif_init, results = all_results_unif_init, beta_cs = all_beta_c_unif_init)

### Load

In [ ]:
rho0 = 0.1
rho1 = 0.9
saved_results = np.load("results/wfp_one_block_rho0={}_rho1={}.npz".format(rho0, rho1), allow_pickle=True)
N, Ks, rho0, rho1, betas_unif_init, all_results_unif_init, all_beta_c_unif_init = saved_results['N'], saved_results['Ks'], saved_results['rho0'].item(), saved_results['rho1'].item(), saved_results['betas'], saved_results['results'].item(), saved_results['beta_cs'].item()

### Plot

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
Bs = sorted(set([B for (K, B) in all_results_unif_init.keys()]))
for B in Bs:
    for K in Ks:
        all_beta_c_ratios = []
        if (K, B) not in all_results_unif_init:
            continue
        beta_cs = all_beta_c_unif_init[(K, B)]
        all_beta_c_ratios.append((log(K/B), beta_cs.mean().item()))
    all_beta_c_ratios = np.array(all_beta_c_ratios)
    ax.scatter(all_beta_c_ratios[:,0], all_beta_c_ratios[:,1], marker='o')
ax.set_xlabel('$\\log(K/(B))$')
ax.set_ylabel('$\\beta_{c, \\rm{th}}$ / $\\beta_{c, \\rm{emp}}$')
fig.savefig('plots/wfp_one_block_rho0={:.1f}_rho1={:.1f}_beta_crit.pdf'.format(rho0, rho1), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
palette = plt.get_cmap('coolwarm')

for K in Ks:
    fig, ax = plt.subplots(figsize=(4, 3))

    Bs_K = sorted(set([B for (K_, B) in all_results_unif_init.keys() if K_ == K]))
    ratios_K = [B / K for B in Bs_K]

    if len(ratios_K) == 1:
        norm = mpl.colors.Normalize(
            vmin=ratios_K[0] - 1e-12,
            vmax=ratios_K[0] + 1e-12
        )
    else:
        norm = mpl.colors.Normalize(vmin=min(ratios_K), vmax=max(ratios_K))

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=palette)
    sm.set_array([])

    for B in Bs_K:
        all_weights_unif_init = all_results_unif_init[(K, B)]

        all_entropies_unif_init = get_entropies(all_weights_unif_init)
        mean_entropies_unif_init = all_entropies_unif_init.mean(dim=0)

        quantiles_entropies_unif_init = torch.quantile(
            all_entropies_unif_init,
            torch.tensor([0.1, 0.5, 0.9], device=all_entropies_unif_init.device),
            dim=0
        )

        ax.plot(
            betas_unif_init,
            mean_entropies_unif_init / log(K),
            color=palette(norm(B / K)),
            label='$B/K={:.3f}$'.format(B / K)
        )

    ax.set_title('$K={}$'.format(K))
    ax.set_xscale('log')
    ax.set_xlabel('$\\beta$')
    ax.set_ylabel('$H/\\log(K)$')

    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label('$B/K$')

    fig.savefig(
        'plots/wfp_one_block_K={}_rho0={:.1f}_rho1={:.1f}_entropies.pdf'.format(K, rho0, rho1),
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()
    plt.close(fig)

## 1-step Hierarchical vary $\rho_0$

### Run

In [ ]:
from math import sqrt, log2
from itertools import product

torch.manual_seed(11)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N = 32
K = 256

rho1 = 0.9
rho0s = [0.1, 0.3, 0.5, 0.7, 0.9]

betas = torch.logspace(0, 3, 201)[1:].to(device)
num_runs = 100
num_steps = 3000

# B = block size, M = number of blocks
block_sizes = [
    K // (2 ** p)
    for p in range(int(log2(K)) + 1)
    if K // (2 ** p) > 1 and K % (K // (2 ** p)) == 0
]

all_results = {}
all_beta_c = {}

configs = list(product(rho0s, block_sizes))

for rho0, B in tqdm(configs, desc="configs"):
    M = K // B

    print(f"K={K}, B={B}, M={M}, rho0={rho0}")

    beta_cs = []
    weights_runs = []

    for run in tqdm(range(num_runs), leave=False, desc="runs"):
        z0 = torch.randn(N, device=device)
        z_block = torch.randn(M, N, device=device)
        eps = torch.randn(K, N, device=device)

        raw_patterns = (
            sqrt(rho0) * z0[None, :]
            + sqrt(rho1 - rho0) * torch.repeat_interleave(z_block, repeats=B, dim=0)
            + sqrt(1.0 - rho1) * eps
        )

        shift, rms, patterns = patterns_shift_and_rms(raw_patterns)

        gram_matrix = patterns @ patterns.T

        with torch.no_grad():
            beta_c = get_critical_beta_gram_uniform_fixed_point(gram_matrix)
            weights = wfp_gram_uniform_init(
                betas,
                gram_matrix,
                num_steps,
                verbose=False,
            )

        beta_cs.append(float(beta_c))
        weights_runs.append(weights.detach().cpu())

    all_results[(rho0, B)] = torch.stack(weights_runs, dim=0)
    all_beta_c[(rho0, B)] = torch.tensor(beta_cs)

In [ ]:
np.savez("results/wfp_one_block_K={}_rho1={}.npz".format(K, rho1), N=N, rho0s=rho0s,  rho1=rho1, betas = betas, results = all_results, beta_cs = all_beta_c)

### Load

In [ ]:
K = 256
rho1 = 0.9
saved_results = np.load("results/wfp_one_block_K={}_rho1={}.npz".format(K, rho1), allow_pickle=True)
N, rho0s, rho1, betas, all_results, all_beta_c = saved_results['N'], saved_results['rho0s'], saved_results['rho1'].item(), saved_results['betas'], saved_results['results'].item(), saved_results['beta_cs'].item()

### Plot

In [ ]:
palette = plt.get_cmap('coolwarm')
fig, axs = plt.subplots(1, len(rho0s), figsize=(5*len(rho0s), 4), sharey=True)
for idx, rho0 in enumerate(rho0s):
    ax = axs[idx]
    for (rho0_, B), weights_runs in all_results.items():
        if rho0_ != rho0:
            continue

        all_entropies = get_entropies(weights_runs)
        mean_entropies = all_entropies.mean(dim=0)
        quantiles_entropies = torch.quantile(all_entropies, torch.tensor([0.1, 0.5, 0.9], device=all_entropies.device), dim=0)

        ax.plot(betas, mean_entropies/ log(K), label='$B={}$'.format(B), color=palette(B/K), zorder=1,  path_effects=[
            patheffects.Stroke(linewidth=2.5, foreground="black"),
            patheffects.Normal()
        ])
        beta_c_emp_quantiles = np.quantile(all_beta_c[(rho0_, B)], [0.05, 0.5, 0.95])
        ax.fill_betweenx([0, 1], beta_c_emp_quantiles[0], beta_c_emp_quantiles[2], linestyle='--', alpha=0.3, color=palette(B/K), zorder=0)
    
    ax.set_title('$\\rho_0={}$'.format(rho0))
    ax.set_xscale('log')
    ax.set_xlabel('$\\beta$')
    if idx == 0:
        ax.set_ylabel('$H/\\log(K)$')
    ax.legend()
fig.savefig(
        'plots/wfp_one_block_K={}_rho1={:.1f}_entropies.pdf'.format(K, rho1),
        dpi=300,
        bbox_inches='tight'
    )
plt.show()

### Gene data

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
gene_df = pd.read_csv("datasets/gene_data.csv")

In [ ]:
print_summary = False

# Choose how many top genes to display
top_n = 20

# -----------------------------
# Gene metadata
# -----------------------------

gene_info = (
    gene_df[["geneId", "gene_symbol"]]
    .drop_duplicates("geneId")
    .set_index("geneId")
)

# -----------------------------
# 1. Expression per lineage
# rows = cell_lineage
# cols = geneId
# values = average expression
# -----------------------------

lineage_expr_raw = pd.pivot_table(
    gene_df,
    index="cell_lineage",
    columns="geneId",
    values="expression",
    aggfunc="mean",
    fill_value=0,
)

gene_ids = lineage_expr_raw.columns.to_numpy()
lineage_ids = lineage_expr_raw.index.to_numpy()

lineage_expr = lineage_expr_raw.copy()
lineage_expr_log = lineage_expr_raw.apply(lambda x: torch.tensor(x.values, dtype=torch.float32).log1p().numpy(), axis=1, result_type="expand")
lineage_expr_log.index = lineage_expr_raw.index
lineage_expr_log.columns = lineage_expr_raw.columns

lineage_expressions = torch.tensor(
    lineage_expr_raw.to_numpy(),
    dtype=torch.float32,
).log1p()


# -----------------------------
# 2. Expression per sample/cell
# rows = sampleId
# cols = geneId
# values = average expression
# -----------------------------

cell_expr_raw = pd.pivot_table(
    gene_df,
    index="sampleId",
    columns="geneId",
    values="expression",
    aggfunc="mean",
    fill_value=0,
)

# same gene order as lineage expression
cell_expr_raw = cell_expr_raw.reindex(columns=gene_ids, fill_value=0)

sample_ids = cell_expr_raw.index.to_numpy()

cell_expressions = torch.tensor(
    cell_expr_raw.to_numpy(),
    dtype=torch.float32,
).log1p()


# -----------------------------
# 3. Metadata per sample/cell
# -----------------------------

metadata_cols = [
    "sampleId",
    "celltype",
    "cell Type Description",
    "cell_lineage",
    "tissue",
    "treatment",
    "immunophenotype",
    "species",
    "strain",
    "sampleCode",
    "RNA.Id",
    "batch",
    "firstPublished",
    "cellOrder",
]

metadata_cols = [c for c in metadata_cols if c in gene_df.columns]

cell_metadata = (
    gene_df[metadata_cols]
    .drop_duplicates("sampleId")
    .set_index("sampleId")
    .loc[cell_expr_raw.index]
)

cell_lineages = cell_metadata["cell_lineage"].to_numpy()
cell_types = cell_metadata["celltype"].to_numpy()


# -----------------------------
# 4. Top expressed genes per lineage
# -----------------------------

def top_genes(row, top_n=20):
    top = row.sort_values(ascending=False).head(top_n)

    out = []
    for gene_id, expr in top.items():
        symbol = gene_info.loc[gene_id, "gene_symbol"] if gene_id in gene_info.index else gene_id
        out.append((gene_id, symbol, expr))

    return out


top_genes_by_lineage = {
    lineage: top_genes(lineage_expr_raw.loc[lineage], top_n=top_n)
    for lineage in lineage_expr_raw.index
}


# -----------------------------
# 5. Top expressed genes per sample/cell
# -----------------------------

top_genes_by_cell = {
    sample_id: top_genes(cell_expr_raw.loc[sample_id], top_n=top_n)
    for sample_id in cell_expr_raw.index
}


# -----------------------------
# 6. Useful lookup dictionaries
# -----------------------------

geneId_to_col = {gene_id: i for i, gene_id in enumerate(gene_ids)}
lineage_to_row = {lineage: i for i, lineage in enumerate(lineage_ids)}
sampleId_to_row = {sample_id: i for i, sample_id in enumerate(sample_ids)}

if print_summary:
    print("lineage_expressions:", lineage_expressions.shape)
    print("cell_expressions:", cell_expressions.shape)

    print("\nExample lineage:")
    example_lineage = lineage_ids[0]
    print(example_lineage)
    print(top_genes_by_lineage[example_lineage][:10])

    print("\nExample sample/cell:")
    example_cell = sample_ids[0]
    print(example_cell)
    print(cell_metadata.loc[example_cell])
    print(top_genes_by_cell[example_cell][:10])

### Per lineage

In [ ]:
shift, rms, patterns = patterns_shift_and_rms(lineage_expressions)
K = patterns.shape[0]
N = patterns.shape[1]
gram_matrix = patterns @ patterns.T

In [ ]:
betas = torch.logspace(-1, 3, 101)

In [ ]:
annealed_fixed_points = annealing_uniform_init(betas, patterns, dt=0.001,num_steps=5000, num_runs=100, verbose=True)

In [ ]:
cosine_similarities = annealed_fixed_points @ patterns.T
cosine_similarities /= annealed_fixed_points.norm(dim=-1, keepdim=True) * patterns.norm(dim=-1, keepdim=True).T

In [ ]:
true_annealed_fixed_points = annealed_fixed_points * rms + shift
true_cosine_similarities = true_annealed_fixed_points @ lineage_expressions.T
true_cosine_similarities /= true_annealed_fixed_points.norm(dim=-1, keepdim=True) * lineage_expressions.norm(dim=-1, keepdim=True).T

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
import matplotlib.cm as cm
color_vmin = cosine_similarities.min().item()
color_vmax = cosine_similarities.max().item()
cmap_norm = Normalize(vmin=color_vmin, vmax=color_vmax)
cmap = cm.coolwarm
for run in range(cosine_similarities.shape[0]):
    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(cosine_similarities[run].cpu(), origin='lower', aspect='auto', cmap=cmap, norm=cmap_norm, extent=[0, K, betas[0].log10().item(), betas[-1].log10().item()])    
    ax.set_xticks(np.arange(K)+0.5, labels=lineage_ids, rotation=90)
    ax.set_xlabel('Lineage')
    ax.set_ylabel('$\\log_{\\rm 10} \\beta$')
    plt.colorbar(cm.ScalarMappable(norm=cmap_norm, cmap=cmap), ax=ax, label='Cosine Similarity')
    plt.show()

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
import matplotlib.cm as cm
color_vmin = true_cosine_similarities.min().item()
color_vmax = true_cosine_similarities.max().item()
cmap_norm = Normalize(vmin=color_vmin, vmax=color_vmax)
cmap = cm.coolwarm
for run in range(true_cosine_similarities.shape[0]):
    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(true_cosine_similarities[run].cpu(), origin='lower', aspect='auto', cmap=cmap, norm=cmap_norm, extent=[0, K, betas[0].log10().item(), betas[-1].log10().item()])    
    ax.set_xticks(np.arange(K)+0.5, labels=lineage_ids, rotation=90)
    ax.set_xlabel('Lineage')
    ax.set_ylabel('$\\log_{\\rm 10} \\beta$')
    plt.colorbar(cm.ScalarMappable(norm=cmap_norm, cmap=cmap), ax=ax, label='Cosine Similarity')
    plt.show()

In [ ]:
avg_cosine_similarities = cosine_similarities.mean(dim=0)
avg_true_cosine_similarities = true_cosine_similarities.mean(dim=0)
color_vmin = avg_cosine_similarities.min().item()
color_vmax = avg_cosine_similarities.max().item()
cmap_norm = Normalize(vmin=color_vmin, vmax=color_vmax)
cmap = cm.coolwarm
fig, ax = plt.subplots(figsize=(4,4))
ax.imshow(avg_cosine_similarities.cpu(), origin='lower', aspect='auto', cmap=cmap, norm=cmap_norm, extent=[0, K, betas[0].log10().item(), betas[-1].log10().item()])    
ax.set_xticks(np.arange(K)+0.5, labels=lineage_ids, rotation=90)
ax.set_xlabel('Lineage')
ax.set_ylabel('$\\log_{\\rm 10} \\beta$')
plt.colorbar(cm.ScalarMappable(norm=cmap_norm, cmap=cmap), ax=ax, label='Cosine Similarity')
plt.show()

In [ ]:
beta_c = get_critical_beta_gram_uniform_fixed_point(gram_matrix)
weights = wfp_gram_dirichlet_init(
    betas,
    gram_matrix,
    num_steps,
    dirichlet_concentration_param=1.0,
    num_runs=100,
    verbose=True,
)

In [ ]:
plt.plot(betas,get_entropies(weights).mean(dim=0), marker='o')
plt.xscale('log')